# 🎧 Techno vs. House – What makes a track danceable?

**Question:** How do electronic music genres (techno, house, deep house, minimal techno …) differ in tempo, energy and danceability – and which audio features go together with danceability?

**Data:** [Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset) (~114,000 tracks, 125 genres, with audio features like `tempo`, `energy`, `danceability`).

**Tools:** Python · pandas · matplotlib · seaborn

## 1. Setup

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
os.makedirs("images", exist_ok=True)

## 2. Load the data

Download `dataset.csv` from Kaggle and put it in a folder `data/` next to this notebook.
(If the file isn't there and `huggingface_hub` is installed, it loads the same data directly from the internet.)

In [2]:
try:
    df = pd.read_csv("data/dataset.csv")
except FileNotFoundError:
    # pip install huggingface_hub
    df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv")

df.shape

(114000, 21)

In [3]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

## 3. Clean the data

In [5]:
# Drop the old index column if present
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Missing values
print(df.isna().sum()[df.isna().sum() > 0])
df = df.dropna()

# Keep one row per track & genre
before = len(df)
df = df.drop_duplicates(subset=["track_id", "track_genre"])
print(f"Removed {before - len(df)} duplicate rows")

artists       1
album_name    1
track_name    1
dtype: int64
Removed 450 duplicate rows


## 4. Focus on electronic genres

In [6]:
# Which electronic genres exist in the data?
sorted(g for g in df["track_genre"].unique() if any(k in g for k in ["techno", "house", "electro", "edm", "dance", "trance", "dub"]))

['chicago-house',
 'dance',
 'dancehall',
 'deep-house',
 'detroit-techno',
 'dub',
 'dubstep',
 'edm',
 'electro',
 'electronic',
 'house',
 'j-dance',
 'minimal-techno',
 'progressive-house',
 'techno',
 'trance']

In [7]:
# TODO: adjust this list to the genres you want to compare
genres = ["techno", "minimal-techno", "detroit-techno", "house", "deep-house", "chicago-house"]
genres = [g for g in genres if g in df["track_genre"].unique()]

edm = df[df["track_genre"].isin(genres)].copy()
edm["track_genre"].value_counts()

track_genre
techno            1000
house              999
chicago-house      998
deep-house         998
detroit-techno     998
minimal-techno     998
Name: count, dtype: int64

## 5. Compare the genres

In [8]:
features = ["tempo", "energy", "danceability", "loudness", "valence", "instrumentalness"]
summary = edm.groupby("track_genre")[features].mean().round(2).sort_values("tempo")
summary

,tempo,energy,danceability,loudness,valence,instrumentalness
track_genre,,,,,,
deep-house,120.89,0.74,0.71,-6.97,0.45,0.24
house,121.12,0.76,0.67,-5.26,0.49,0.04
chicago-house,123.91,0.73,0.77,-9.16,0.59,0.48
minimal-techno,124.75,0.68,0.73,-9.78,0.28,0.75
detroit-techno,126.60,0.71,0.72,-11.09,0.47,0.70
techno,128.26,0.75,0.68,-8.08,0.32,0.54


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ["tempo", "energy", "danceability"]):
    sns.boxplot(data=edm, x="track_genre", y=col, order=summary.index, ax=ax)
    ax.set_title(col.capitalize())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("images/genre_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

**What I see:**

Techno is the fastest genre here with about 128 BPM on average, then Detroit techno (127) and minimal techno (125). House and deep house are the slowest at around 121 BPM.

Energy is pretty similar for all of them (0.68 to 0.76). I expected techno to have more energy than house but they are almost the same (0.75 vs 0.76). Minimal techno has the lowest energy.

The most danceable genre is Chicago house (0.77), even though it's slower than techno. House and techno are actually the least danceable (0.67 and 0.68). From DJing this kind of makes sense to me, Chicago house has more swing and groove, techno is more about pushing intensity.

## 6. What goes together with danceability?

In [10]:
corr = edm[features + ["popularity"]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of audio features (electronic genres)")
plt.savefig("images/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

corr["danceability"].drop("danceability").sort_values()

energy             -0.191082
tempo              -0.162341
loudness           -0.154150
popularity         -0.135405
instrumentalness    0.138864
valence             0.230516
Name: danceability, dtype: float64

In [11]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=edm.sample(min(3000, len(edm)), random_state=0),
                x="tempo", y="danceability", hue="track_genre", alpha=0.5, s=20)
plt.title("Tempo vs. danceability")
plt.savefig("images/tempo_vs_danceability.png", dpi=150, bbox_inches="tight")
plt.show()

**What I see:**

Valence (how happy/positive a track sounds) has the strongest positive correlation with danceability (0.23), then instrumentalness (0.14).

What surprised me: energy (-0.19), tempo (-0.16) and loudness (-0.15) are negatively correlated with danceability. So faster or harder tracks are not more danceable, at least not by Spotify's measure.

But all correlations are weak (under 0.25), so no feature explains danceability alone. In the scatter plot you can also see that danceability is spread out at every tempo.

## 7. Bonus: simple regression – what predicts danceability?

In [12]:
# pip install statsmodels
import statsmodels.formula.api as smf

model = smf.ols("danceability ~ tempo + energy + loudness + valence + instrumentalness", data=edm).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           danceability   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.163
Method:                 Least Squares   F-statistic:                     234.4
Date:                Fri, 13 Nov 2026   Prob (F-statistic):          2.88e-229
Time:                        19:08:56   Log-Likelihood:                 5540.5
No. Observations:                5991   AIC:                        -1.107e+04
Df Residuals:                    5985   BIC:                        -1.103e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.8755      0.014     61.240      0.000       0.847       0.903
tempo               -0.0011   9.35e-05    -12.264      0.000      -0.001      -0.001
energy              -0.1311      0.010    -13.320      0.000      -0.150      -0.112
loudness             0.0012      0.001      2.223      0.026       0.000       0.002
valence              0.1295      0.005     25.331      0.000       0.119       0.139
instrumentalness     0.0649      0.004     16.630      0.000       0.057       0.073
==============================================================================
Omnibus:                      256.521   Durbin-Watson:                   1.830
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              319.072
Skew:                          -0.461   Prob(JB):                     5.18e-70
Kurtosis:                       3.654   Cond. No.                     1.57e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.57e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**Interpretation:**

All five variables are significant (p < 0.05). Valence (+0.13) and instrumentalness (+0.06) increase danceability, energy (-0.13) and tempo decrease it (about -0.01 per 10 BPM). Loudness has almost no effect.

R² is only 0.16, so the model explains around 16% of the variation in danceability. I think most of it comes from things like groove, swing and percussion patterns which these features don't really capture.

One limitation: these are Spotify's own audio features and we don't know exactly how they are calculated. Also this only shows correlations, not causes.

## 8. Conclusion

- Techno is the fastest (~128 BPM), house and deep house the slowest (~121 BPM)
- Chicago house is the most danceable genre, so faster and harder doesn't mean more danceable
- Valence is the feature most linked to danceability, energy and tempo are slightly negative
- The regression only explains about 16% (R² = 0.16), rhythm and groove seem to matter more than tempo or energy
- Next I want to analyse my own DJ tracks with librosa (BPM, key) and maybe try predicting the genre from the audio features